In [2]:
import tensorflow as tf
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from tensorflow.keras import optimizers

In [3]:
def adjacency_matrix_to_mol(matrix, atom_types):
  mol = Chem.RWMol()
  atom_map = {}

  if len(matrix) != len(atom_types):
    raise ValueError("NUMBER OF ATOM TYPES DOES NOT MATCH MATRIX DIMENSIONS")

  # Step 1: Add Atoms
  for i, atom_num in enumerate(atom_types):
    atom = Chem.Atom(atom_num)
    mol_idx = mol.AddAtom(atom)
    atom_map[i] = mol_idx

  # Step 2: Add Bonds
  for i, row in enumerate(matrix):
    for j, value in enumerate(row):
      if i < j and value != 0:
        if value == 1:
          bond_type = Chem.BondType.SINGLE
        elif value == 2:
          bond_type = Chem.BondType.DOUBLE
        elif value == 3:
          bond_type = Chem.BondType.TRIPLE
        elif value == 4:
          bond_type = Chem.BondType.AROMATIC
        else:
          raise ValueError(f"INVALID BOND TYPE DETECTED: {value}")

        mol.addBond(atom_map[i], atom_map[j], bond_type)

  return mol

In [4]:
def validity_reward(mol):
    if mol is None:
      return 0
    # Sanitize
    try:
      Chem.SanitizeMol(mol)
    except Exception:
      return 0
    # Check Kekulization
    try:
      Chem.Kekulize(mol, clearAromaticFlags = True)
    except Exception:
      return 0
    # Check Valency
    for atom in mol.GetAtoms():
      explicit_valence = atom.GetExplicitValence()
      if explicit_valence > atom.GetTotalValence():
        return 0
    return 1

In [5]:
def morg_fp(mol):
  fp_gen = rdFingerprintGenerator.GetMorganGenerator(
                                                    radius = 2,
                                                    fpSize = 2048
                                                    )
  return fp_gen.GetFingerprint(mol)

In [6]:
def uniqueness_reward(generated_mols, curr_mol):
  gen_fps = [morg_fp(mol) for mol in generated_mols]
  unique_gen = set(gen_fps)
  if validity_reward(curr_mol) == 0:
      return 0
  curr_fp = morg_fp(curr_mol)
  if curr_fp in unique_gen:
      return 0
  else:
      return 1

In [7]:
def novelty_reward(curr_mol, train_mols, generated_mols):
    train_fps = [morg_fp(mol) for mol in train_mols]
    unique_train = set(train_fps)
    if validity_reward(curr_mol) == 0:
        return 0
    elif uniqueness_reward(generated_mols, curr_mol) == 0:
        return 0
    curr_fp = morg_fp(curr_mol)
    if curr_fp in unique_train:
        return 0
    else:
        return 1

In [8]:
def reinforce_train(generator, dataset, num_epoch = 1000):
    optimizer = optimizers.Adam(learning_rate = 0.001)
    generated_mols = []
    for epoch in range(num_epoch):
        with tf.GradientTape() as tape:
            z = tf.random.normal((1, 16))  
            adj_matrix, node_features = generator(z) 
            atom_types = set(adj_matrix[1])
            mol = adjacency_matrix_to_mol(adj_matrix, atom_types)
            generated_mols.append(mol)

            validity = validity_reward(mol)
            uniqueness = uniqueness_reward(mol, generated_mols)
            novelty = novelty_reward(mol, train_mols, generated_mols)

            total_reward = validity + uniqueness + novelty
            
            # Compute loss as negative reward (maximize reward)
            loss = -total_reward  

        # Compute gradients and update generator
        gradients = tape.gradient(loss, generator.trainable_variables)
        optimizer.apply_gradients(zip(gradients, generator.trainable_variables))

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Reward: {total_reward.numpy()}")
